In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

In [2]:
import re
import torch
import numpy as np
import torch.nn as nn
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from models.lstm_model import LSTM_Model
from utils import EarlyStopping, get_device,EpochTrainer,tokenize,encode_text, NLP_data_cleaning
from datasets import load_dataset
import kagglehub
from pathlib import Path

import pandas as pd

# Parameters

In [3]:
model_output_filename = "../../checkpoints/sentiment_checkpoint.pt"

IsDownload_data = True
data_src_path = "zeroshot/twitter-financial-news-sentiment"
data_src_path2 = "sbhatti/financial-sentiment-analysis"
data_save_path = '../../data/news_sentiment.csv'

## model config

In [4]:
input_size = None
batch_size = 32
epochs = 100
hidden_size = 128
num_output = 3
num_layers = 1
dropout = 0.3
bidirectional = True
patience = 5
test_size = 0.2
lr = 0.01

PAD_IDX = 0
UNK_IDX = 1
embedding_dim = 200
max_seq_len = 30

In [5]:
eval_method = 'Accuracy'

In [6]:
label_map = {
    "negative": 0,
    "positive": 1,
    "neutral": 2,
}

# Load and prepare training data

## Download dataset and save as csv

### Function to download from huggingface

In [7]:
def hugface_download(path):
    
    dataset = load_dataset( path , split="train")    
    
    df = pd.DataFrame({
        'text': [row['text'] for row in dataset],
        'label': [row['label'] for row in dataset]
    })   

    return df
   

### Function to download from kaggle

In [8]:
def kaggle_download(path):
    
    dataset_path = kagglehub.dataset_download(data_src_path2)
    csv_path = next(Path(dataset_path).rglob("*.csv"))

    df = pd.read_csv(csv_path)
    
    df.columns = ['text','label']
    
    return df
    

### Function to solve data label imbalance

In [9]:
def solve_data_imbalance(df):
    
    min_count = int( (df['label'].value_counts().min() ) * 1.2)
    
    df_lists = []

    
    for lb in df['label'].unique():

        count = len(df[df['label'] == lb])
        sample_size = count if count < min_count else min_count        
        
        sample_df = df[df['label'] == lb].sample(n = sample_size, random_state=42)
        
        df_lists.append(sample_df)
    
    df_balanced = pd.concat(df_lists).sample(frac=1, random_state=42).reset_index(drop=True)

    return df_balanced   
    

### Download Data

In [10]:
if IsDownload_data:

    dataset1 = hugface_download(data_src_path)
    dataset1['source'] = 'twitter'
    
    dataset2 = kaggle_download(data_src_path2)
    dataset2['source'] = 'kaggle'
    

    dataset2["label"] = (dataset2["label"]
                            .astype(str)
                            .str.strip()
                            .str.lower()
                            .map(label_map)    
                        )

    df_all = pd.concat([dataset1,dataset2],ignore_index=True)
    df_all["text"] = df_all["text"].astype(str).str.strip()
    df_all = df_all[df_all["text"] != ""]

    df_all = df_all.drop_duplicates(subset=["text"]).reset_index(drop=True)      


    df_all.to_csv( data_save_path , index=False)

## Load Downloaded dataset csv

In [11]:
df = pd.read_csv(data_save_path)


df = df.dropna(subset=['text', 'label']).copy()
df = df[df['text'] != ""]

df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(int)

## Check Data distribution

In [12]:
# df = df[df['source'] == 'kaggle']

df = solve_data_imbalance(df)

In [13]:
print('Distribution: \n')
print('='*50, df['label'].value_counts().sort_index())

Distribution: 

================================================== label
0    2034
1    2440
2    2440
Name: count, dtype: int64


## Preprocess dataset

Goal: Convert raw text sentences into fixed-length numeric sequences for the model
Overview:
1. Tokenize : Convert sentence to lowercase, remove punctuation, split into words
2. Build Vocab : Count word frequency across entire dataset, keep words appearing >=2 times,
   assign each word a unique ID (reserve 0 for PAD, 1 for UNK)
3. Encode : Replace each word with its vocab ID (unknown words → UNK_IDX)
4. Pad/Truncate : Ensure all sequences are length 40 (pad short ones, truncate long ones)
       
Result: All sentences become fixed-length arrays of integers, ready for embedding layer
Example flow:
- Raw text:        "Strong earnings! Q3 beat."
- After tokenize:  ["strong", "earnings", "q3", "beat"]
- Vocab mapping:   {"<PAD>": 0, "<UNK>": 1, "strong": 2, "earnings": 3, "beat": 4, ...}
- After encode:    [2, 3, 1, 4]                    (q3 not in vocab → use UNK_IDX=1)
- After pad (40):  [2, 3, 1, 4, 0, 0, 0, ..., 0]   (pad to 40 with zeros)

In [14]:
texts = df['text'].tolist()
labels = df['label'].tolist()

### Data cleaning

In [15]:
def data_cleaning_chk():

    for id, item in enumerate(texts):
        before = item 
        after = NLP_data_cleaning(item)
        label = [k for k,v in label_map.items() if v == labels[id]][0]
    
        if before.lower().strip() != after.strip().lower():
            print('Before: ',before)
            print('After:  ',after)
            print(label)
            print('\n', '='*50)       

In [16]:
# data_cleaning_chk()

In [17]:
texts = [ NLP_data_cleaning(text) for text in texts]

### Split data into training and testing

In [18]:
texts_train, texts_test, labels_train, labels_test = train_test_split(
    texts,
    labels,
    test_size=test_size,
    random_state=42,
    stratify=labels,
)

### Build vocabulary

In [19]:
counter = Counter()

for text in texts_train:
    counter.update(tokenize(text))

In [20]:
vocab = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}

for token, freq in counter.items():
    
    if freq >= 2:
        vocab[token] = len(vocab)

print(f"Vocab size: {len(vocab)}")

Vocab size: 4137


### Convert to sequences

In [21]:
X_train = np.array([encode_text(text, vocab, UNK_IDX, PAD_IDX, max_seq_len ) for text in texts_train], dtype=np.int64)
y_train = np.array(labels_train, dtype=np.int64)

X_test = np.array([encode_text(text, vocab, UNK_IDX, PAD_IDX, max_seq_len) for text in texts_test], dtype=np.int64)
y_test = np.array(labels_test, dtype=np.int64)

In [22]:
X_train_tensor = torch.tensor(X_train, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Prepare Model

## Prepare config of the model

In [23]:
preprocess_config = {
    "seq_length": max_seq_len,
    "padding_idx": PAD_IDX,
    "unk_idx": UNK_IDX,
    "task_type": "classification",
}

In [24]:
model_config = {
    "input_size": input_size,
    "hidden_size": hidden_size,
    "num_output": num_output,
    "num_layers": num_layers,
    "dropout": dropout,
    "bidirectional": bidirectional,
    "vocab_size": len(vocab),
    "embedding_dim": embedding_dim,
    "padding_idx": PAD_IDX,
}

## Get Model

In [25]:
device = get_device()

In [26]:
model = LSTM_Model(**model_config).to(device)

In [27]:
optimizer = torch.optim.Adam(model.parameters(),  lr = lr)

In [28]:
class_counts = np.bincount(y_train)  # [count_class0, count_class1, count_class2]
class_weights = len(y_train) / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

## Early stopping and checkpoint setup

In [29]:
early_stopping = EarlyStopping(
    patience = patience,
    path = model_output_filename,
    checkpoint_data={
        "model_config": model_config,
        "preprocess_config": preprocess_config,
        "vocab": vocab,
        "label_map": label_map,
        "max_len": preprocess_config["seq_length"],
    },
)

# Loop epochs and train model

In [30]:
epoch_trainer = EpochTrainer(
    model = model,
    early_stopping = early_stopping,
    device = device,
    optimizer = optimizer,
    criterion = criterion,
    eval_method = eval_method
)

In [31]:
for epoch in range(epochs):

    avg_train_loss, avg_val_loss, result = epoch_trainer(train_loader , test_loader )

    for key, value in result.items():        
        
        print(f"Epoch {epoch} | Train Loss: {avg_train_loss / len(train_loader):.4f} | Val Loss: {avg_val_loss / len(test_loader):.4f} | {key}: {value:.4f}")
    
    
    # ==================== Early Stopping Check ====================

    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        
        print("Early stopping triggered! Training stopped.")
        
        break
        

Epoch 0 | Train Loss: 0.0047 | Val Loss: 0.0164 | Accuracy: 0.7014
Epoch 1 | Train Loss: 0.0024 | Val Loss: 0.0176 | Accuracy: 0.7158
Epoch 2 | Train Loss: 0.0011 | Val Loss: 0.0217 | Accuracy: 0.7202
Epoch 3 | Train Loss: 0.0005 | Val Loss: 0.0296 | Accuracy: 0.7086
Epoch 4 | Train Loss: 0.0003 | Val Loss: 0.0306 | Accuracy: 0.7216
Epoch 5 | Train Loss: 0.0002 | Val Loss: 0.0355 | Accuracy: 0.7180
Early stopping triggered! Training stopped.
